In [54]:
import pandas as pd
import numpy as np
import nltk

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')  # Ensures the LookupError is resolved
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to C:\Users\IQBAL
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\IQBAL
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [55]:
# Load your dashboard data
dashboard_data = pd.read_csv("data/raw/dashboard_data.csv")

# Load Bitext dataset
bitext_data = pd.read_csv("data/raw/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

# Filter relevant intents (e.g., ORDER, FEEDBACK)
support_intents = bitext_data[bitext_data['category'].isin(['ORDER', 'FEEDBACK'])]

In [56]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return ' '.join(tokens)

dashboard_data['processed_text'] = dashboard_data['ticket_type'].apply(preprocess_text)
support_intents['processed_instruction'] = support_intents['instruction'].apply(preprocess_text)

# Merge datasets (simple concatenation for now)
combined_data = pd.concat([dashboard_data[['processed_text', 'ticket_type', 'status', 'scheduled_date']], 
                          support_intents[['processed_instruction', 'intent']]], 
                         axis=0, ignore_index=True)

C:\Users\IQBAL SINGH\AppData\Local\Temp\ipykernel_9680\2520481976.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  support_intents['processed_instruction'] = support_intents['instruction'].apply(preprocess_text)


In [57]:
def recognize_intent(user_input):
    processed_input = preprocess_text(user_input)
    # Simple keyword-based intent matching
    for index, row in combined_data.iterrows():
        # Check if processed_text is not NaN and contains the input
        if (pd.notna(row['processed_text']) and processed_input in str(row['processed_text'])) or \
           (pd.notna(row['processed_instruction']) and processed_input in str(row['processed_instruction'])):
            # Check if the row has a status (from dashboard_data)
            if 'status' in row and pd.notna(row['status']):
                if row['status'] == 'scheduled':
                    return 'scheduled_query'
                elif row['status'] == 'resolved':
                    return 'resolved_query'
                else:
                    return 'ticket_query'
            # If no status, fall back to intent from Bitext dataset
            return row['intent'] if 'intent' in row and pd.notna(row['intent']) else 'ticket_query'
    return 'unknown'

# Test intent recognition
test_input = "I need help with my inverter not functioning"
intent = recognize_intent(test_input)
print(f"Detected intent: {intent}")

Detected intent: unknown


In [58]:
def generate_response(intent, user_input):
    # Find the row in dashboard_data that matches the user_input (if applicable)
    processed_input = preprocess_text(user_input)
    matching_row = dashboard_data[dashboard_data['processed_text'].str.contains(processed_input, na=False)]

    # Rule-based responses based on intent
    if intent == 'cancel_order':
        response = "To cancel your order, please log into our portal and navigate to the 'Cancel Order' section."
    elif intent == 'ticket_query':
        response = f"We've noted your issue: {user_input}. A technician will assist you soon."
    elif intent == 'scheduled_query':
        if not matching_row.empty:
            scheduled_date = matching_row.iloc[0]['scheduled_date']
            response = f"Your {user_input} is scheduled for {scheduled_date}. We’ll send a reminder closer to the date."
        else:
            response = "It looks like you have a scheduled service, but I don’t have the details. Please check the portal."
    elif intent == 'resolved_query':
        response = f"Your issue ({user_input}) has already been resolved. If you have further questions, please let us know!"
    elif intent == 'feedback':
        response = "Thank you for your feedback! We’ll pass it along to our team to improve our services."
    elif intent == 'order':
        response = "To place an order, please visit our website or contact our support team at support@solarenergy.com."
    else:
        # Default response for unknown intents
        response = "I'm sorry, I didn’t quite understand that. Could you please rephrase your request?"
    
    return response

# Test response generation
response = generate_response(intent, test_input)
print(f"Response: {response}")

Response: I'm sorry, I didn’t quite understand that. Could you please rephrase your request?


In [ ]:
def chatbot():
    print("Welcome to Solar Support Chatbot! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        intent = recognize_intent(user_input)
        response = generate_response(intent, user_input)
        print(f"Bot: {response}")

# Run the chatbot
chatbot()

Welcome to Solar Support Chatbot! Type 'exit' to quit.


You:  hello


Bot: I'm sorry, I didn’t quite understand that. Could you please rephrase your request?


You:  hi


Bot: To cancel your order, please log into our portal and navigate to the 'Cancel Order' section.


In [8]:
import osKARAGA
import nltk

nltk.data.path.append(os.path.expanduser('~') + '/nltk_data')
nltk.download('punkt')
#debugging

[nltk_data] Downloading package punkt to C:\Users\IQBAL
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [9]:
import nltk
print(nltk.data.path)
#debubgging

['C:\\Users\\IQBAL SINGH/nltk_data', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\nltk_data', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\share\\nltk_data', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\lib\\nltk_data', 'C:\\Users\\IQBAL SINGH\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'C:\\Users\\IQBAL SINGH/nltk_data']


In [52]:
TO KABHI KARAGA..

SyntaxError: invalid syntax (2046000280.py, line 1)

In [ ]:
TRAIN KO DEKHNA padega kaise hoga isami nahi hai wo bs direct de rkha hai 

In [ ]:
and hugging face ki website p kayi saare models free mai hai 

In [ ]:
SUNA AGAR KOI MAHARI FIELD SE RELATED MILA TO LALA OR CHECK KARKA SEND KARIYO OR  SUNA PRIDICT KA BHI CODE SEPehle TRAIN HONE TOU DEO

In [ ]:
ye kya HAI?

In [ ]:
OK KARLA JO BHI KARSAKA hmm 

In [36]:
# Print the columns and first few rows of the Bitext dataset
print(bitext_data.columns)
print(bitext_data.head())

#debugging

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')
   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  
0  cancel_order  I've understood you have a question regarding ...  
1  cancel_order  I've been informed that you have a question ab...  
2  cancel_order  I can sense that you're seeking assistance wit...  
3  cancel_order  I understood that you need assistance with can...  
4  cancel_order  I'm sensitive to the fact that you're facing f...  


In [37]:
import os

# List files in data/raw/
print(os.listdir("data/raw/"))

#debugging

['Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv', 'dashboard_data.csv']
